[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com)

# Lesson 9 — File I/O

**Module 1 — Python Fundamentals** | ⏱ 20 min

Reading and writing files is one of the most common tasks in real-world programming. Python provides clean, simple tools for working with text files, CSV data, and JSON — the three most common file formats you will encounter. The `with` statement (context manager) is the correct way to work with files because it guarantees the file is properly closed even if an error occurs.

## Learning Objectives
- Open files with `open()` and understand the different modes
- Use the `with` statement as the standard approach to file handling
- Read files with `read()`, `readline()`, and `readlines()`
- Write and append to files
- Read and write CSV files using the `csv` module
- Parse and serialise JSON data with the `json` module
- Use `pathlib.Path` for modern path handling
- Handle `FileNotFoundError` and other file-related exceptions

## Opening Files and the with Statement

The built-in `open()` function opens a file and returns a **file object**. You must always close a file when you are done with it — otherwise you risk data loss or resource leaks. The `with` statement (context manager) handles this automatically: the file is closed as soon as the `with` block exits, regardless of whether an exception occurred. This is the only correct way to work with files in Python.

In [ ]:
# First, let's create a sample file to work with using %%writefile magic
# The %%writefile cell magic writes the cell contents directly to a file
pass  # Placeholder — the actual file creation is in the next cell

In [ ]:
%%writefile sample_notes.txt
Meeting Notes - Project Kickoff
Date: 2024-01-15
Attendees: Alice, Bob, Carol

Action Items:
1. Set up development environment
2. Create project repository
3. Schedule weekly standup
4. Review technical specifications

Next meeting: 2024-01-22

In [ ]:
# File open modes:
# 'r'  - read text (default)
# 'w'  - write text (creates new or overwrites existing)
# 'a'  - append text (adds to end without overwriting)
# 'rb' - read binary
# 'wb' - write binary
# 'r+' - read and write

# Reading a whole file with read()
with open("sample_notes.txt", "r") as file:
    content = file.read()  # Reads the entire file as one string
    print(content)

# The file is automatically closed when the 'with' block ends
# file.read() here would raise ValueError: I/O operation on closed file

In [ ]:
# Reading line by line — memory-efficient for large files
print("First 4 lines:")
with open("sample_notes.txt", "r") as file:
    for line_num, line in enumerate(file, start=1):
        print(f"  {line_num}: {line}", end="")  # line already includes \n
        if line_num >= 4:
            break

print("\n")

# readline() — read one line at a time
with open("sample_notes.txt", "r") as file:
    first_line  = file.readline()  # Reads up to and including \n
    second_line = file.readline()
    print(f"Line 1: {first_line.strip()!r}")
    print(f"Line 2: {second_line.strip()!r}")

# readlines() — read all lines into a list
with open("sample_notes.txt", "r") as file:
    lines = file.readlines()
print(f"Total lines: {len(lines)}")
print(f"Last line: {lines[-1].strip()!r}")

## Writing and Appending to Files

Opening a file with mode `'w'` creates a new file or **overwrites** an existing one completely. Mode `'a'` (append) adds new content to the end of the file without removing existing content. Always use the `with` statement when writing — if your program crashes before the file is closed, data may not be flushed to disk. Use `encoding='utf-8'` to handle international characters correctly.

In [ ]:
# Writing to a file
log_entries = [
    "2024-01-15 10:00 - Application started",
    "2024-01-15 10:05 - User 'alice' logged in",
    "2024-01-15 10:12 - Processed 42 records",
    "2024-01-15 10:45 - User 'alice' logged out",
]

with open("app.log", "w", encoding="utf-8") as logfile:
    for entry in log_entries:
        logfile.write(entry + "\n")  # write() does NOT add a newline automatically

print("Log file written.")

# Verify by reading it back
with open("app.log", "r") as logfile:
    print(logfile.read())

# Appending — add new entries without erasing the existing log
new_entries = [
    "2024-01-15 14:00 - Scheduled backup completed",
    "2024-01-15 17:30 - Application stopped",
]
with open("app.log", "a", encoding="utf-8") as logfile:
    for entry in new_entries:
        logfile.write(entry + "\n")

# Count total entries
with open("app.log") as f:
    total = sum(1 for line in f if line.strip())  # Count non-empty lines
print(f"Total log entries: {total}")

## CSV Files

CSV (Comma-Separated Values) is the most common format for tabular data. Python's `csv` module handles the complexity of CSV formatting correctly — including quoting fields that contain commas, handling different delimiters, and managing newlines. `csv.DictReader` is particularly useful because it maps each row to a dictionary with column names as keys, making the data self-describing and easy to work with.

In [ ]:
import csv

# Write a CSV file using csv.writer
employees = [
    ["id", "name",          "department",  "salary"],  # Header row
    [1001,  "Alice Johnson", "Engineering", 95000],
    [1002,  "Bob Smith",     "Marketing",   72000],
    [1003,  "Carol Lee",     "Engineering", 98000],
    [1004,  "David Brown",   "HR",           65000],
    [1005,  "Eve Wilson",    "Engineering", 105000],
]

with open("employees.csv", "w", newline="", encoding="utf-8") as csvfile:
    # newline="" is required on Windows to prevent double newlines
    writer = csv.writer(csvfile)
    writer.writerows(employees)  # Write all rows at once

print("CSV file written.")

# Read with csv.reader
print("\nReading with csv.reader:")
with open("employees.csv", "r", encoding="utf-8") as csvfile:
    reader = csv.reader(csvfile)
    header = next(reader)  # Read and skip the header row
    print(f"Columns: {header}")
    for row in reader:
        print(f"  {row}")

In [ ]:
import csv

# DictReader — maps each row to a dict with column names as keys
print("Engineering salaries (via DictReader):")
total_salary = 0
eng_count = 0

with open("employees.csv", "r", encoding="utf-8") as csvfile:
    reader = csv.DictReader(csvfile)  # Automatically reads header
    for row in reader:
        if row["department"] == "Engineering":
            salary = int(row["salary"])
            total_salary += salary
            eng_count += 1
            print(f"  {row['name']:<15} ${salary:>8,}")

if eng_count:
    print(f"Average Engineering salary: ${total_salary // eng_count:,}")

# DictWriter — write dicts to CSV
new_employees = [
    {"id": 1006, "name": "Frank Garcia", "department": "Marketing", "salary": 78000},
    {"id": 1007, "name": "Grace Kim",    "department": "Engineering", "salary": 99000},
]
with open("new_hires.csv", "w", newline="", encoding="utf-8") as csvfile:
    fieldnames = ["id", "name", "department", "salary"]
    writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
    writer.writeheader()       # Write the header row
    writer.writerows(new_employees)
print("\nNew hires CSV written.")

## JSON Files

JSON (JavaScript Object Notation) is the standard format for web APIs and configuration files. Python's `json` module converts between Python objects and JSON text. The four key functions are: `json.dump()` (write to file), `json.load()` (read from file), `json.dumps()` (convert to string), and `json.loads()` (parse from string). Python dicts map to JSON objects, lists to JSON arrays, and Python's `None` maps to JSON's `null`.

In [ ]:
import json

# Python data to write as JSON
app_config = {
    "app_name": "DataPipeline",
    "version": "2.1.0",
    "debug": False,
    "database": {
        "host": "localhost",
        "port": 5432,
        "name": "pipeline_db",
        "ssl": True
    },
    "allowed_formats": ["csv", "json", "parquet"],
    "max_retries": 3,
    "timeout": None  # null in JSON
}

# Write to file — indent=2 makes it human-readable
with open("config.json", "w", encoding="utf-8") as jsonfile:
    json.dump(app_config, jsonfile, indent=2)

print("config.json written. Contents:")
with open("config.json", "r") as f:
    print(f.read())

In [ ]:
import json

# Read from file
with open("config.json", "r", encoding="utf-8") as jsonfile:
    loaded_config = json.load(jsonfile)  # Returns a Python dict

print(f"App: {loaded_config['app_name']} v{loaded_config['version']}")
print(f"Database host: {loaded_config['database']['host']}:{loaded_config['database']['port']}")
print(f"Allowed formats: {', '.join(loaded_config['allowed_formats'])}")
print(f"Timeout: {loaded_config['timeout']} (type: {type(loaded_config['timeout']).__name__})")  # None

# json.dumps() / json.loads() — work with strings (useful in APIs)
api_response_str = '{"status": "success", "user_id": 42, "token": "abc123"}'
response = json.loads(api_response_str)  # String -> dict
print(f"\nAPI Response: {response}")
print(f"User ID: {response['user_id']}")

# Convert back to JSON string
compact_json = json.dumps(response, separators=(",", ":"))  # No spaces
pretty_json  = json.dumps(response, indent=2)
print(f"Compact: {compact_json}")

## pathlib.Path — Modern Path Handling

`pathlib.Path` (introduced in Python 3.4) provides an object-oriented interface for file system paths. It is cleaner and more portable than string-based path manipulation using `os.path`. Paths support the `/` operator for joining, and provide methods for checking existence, reading/writing, listing directories, and more. It is the modern preferred approach for all path-related operations.

In [ ]:
from pathlib import Path

# Creating Path objects
current_dir = Path(".")
home_dir = Path.home()

print(f"Current directory: {current_dir.resolve()}")
print(f"Home directory: {home_dir}")

# Building paths with the / operator — platform-independent!
data_dir = current_dir / "data"
config_file = current_dir / "config.json"

# Checking existence
print(f"\nconfig.json exists: {config_file.exists()}")
print(f"data/ exists: {data_dir.exists()}")

# Path properties
path = Path("employees.csv")
print(f"\nFile: {path}")
print(f"  Name:      {path.name}")        # employees.csv
print(f"  Stem:      {path.stem}")        # employees (name without extension)
print(f"  Suffix:    {path.suffix}")      # .csv
print(f"  Parent:    {path.parent}")      # . (current directory)
print(f"  Absolute:  {path.resolve()}")  # Full absolute path

# Read a file with Path.read_text()
content = config_file.read_text(encoding="utf-8")
print(f"\nFirst 50 chars of config.json: {content[:50]}...")

## Handling File Exceptions

File operations are prone to errors: the file might not exist, you might not have permission to read it, or the disk might be full. Python raises specific exceptions for each type of error. The most common is `FileNotFoundError`. It is important to catch only the specific exceptions you expect and to handle them meaningfully — not with a bare `except:` that silences all errors.

In [ ]:
from pathlib import Path
import json

# Handling FileNotFoundError gracefully
def load_config(filepath):
    """Load a JSON config file, returning a default config if file not found."""
    path = Path(filepath)
    try:
        with open(path, "r", encoding="utf-8") as f:
            return json.load(f)
    except FileNotFoundError:
        print(f"Warning: {filepath} not found. Using default config.")
        return {"debug": False, "log_level": "INFO"}  # Sensible defaults
    except json.JSONDecodeError as e:
        print(f"Error: {filepath} contains invalid JSON: {e}")
        return None
    except PermissionError:
        print(f"Error: No permission to read {filepath}")
        return None

config = load_config("config.json")
print(f"Loaded config: {list(config.keys())}")

missing = load_config("nonexistent_config.json")
print(f"Missing config fallback: {missing}")

# Safe file writing with error handling
def write_report(data, filepath):
    """Write data to a file, creating parent directories if needed."""
    path = Path(filepath)
    path.parent.mkdir(parents=True, exist_ok=True)  # Create dirs if needed
    try:
        with open(path, "w", encoding="utf-8") as f:
            f.write(data)
        print(f"Report written to {path}")
        return True
    except OSError as e:
        print(f"Failed to write report: {e}")
        return False

write_report("Sales Report\nTotal: $50,000", "reports/summary.txt")
write_report("Error log", "logs/errors.txt")

## Practice Exercises

1. Write a function `word_frequency(filepath)` that reads a text file, splits it into words (lowercased, punctuation stripped), and returns a dictionary mapping each unique word to its frequency. Test it on `sample_notes.txt`.
2. Read the `employees.csv` file and use `csv.DictReader` to calculate the average salary per department. Print the results sorted by department name.
3. Write a function `update_config(filepath, updates)` that loads a JSON config file, merges in the `updates` dict (overwriting existing keys), and saves the file back. Test by adding a `"last_modified"` key.
4. Use `pathlib.Path` to write a script that lists all `.csv` and `.json` files created in this lesson (in the current directory), prints their name, size in bytes, and the first 30 characters of their content.